In [1]:
from nltk.corpus import stopwords#停用词库
import nltk
from nltk.wsd import lesk
from nltk.corpus import wordnet as wn
from utils import clean_str, loadWord2Vec
from nltk.stem.porter import PorterStemmer
poter_stemme=PorterStemmer()
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
print(stop_words)


{'its', 'mightn', 'down', 'when', "wasn't", 'weren', 'does', 'them', 'being', 'by', 'with', 'she', 'below', "you've", 'over', 'shan', "haven't", "you'll", 'had', 'why', 'can', 'himself', 's', 'into', 'myself', 'his', 'an', 'hers', 'other', 'or', 'how', 'needn', 'mustn', 'him', "doesn't", "mightn't", 'too', 'couldn', 'after', 'where', 'to', 'don', 'isn', 'was', 'y', "shouldn't", 'he', 'for', 'more', 'nor', 'as', 'her', "it's", 'all', 'aren', "couldn't", 'are', 'i', 'out', 'now', 'if', 'you', 'but', 'same', 'they', 'm', 'ourselves', 'should', 'll', "don't", 'my', "won't", 'so', 'and', 'very', 'has', 'herself', 'during', 'themselves', 'before', 'between', 'not', "wouldn't", 'few', 'above', 'won', 'who', 'just', 'that', "that'll", 'be', 'most', 'because', 'those', 'again', 'ours', 'have', 'only', 'itself', 'these', "you'd", 'your', 'from', 'here', 'such', 'hadn', 'were', 'this', "didn't", 'yours', 'both', 'yourself', 't', 'what', "aren't", 'yourselves', 'doesn', 'off', 'whom', 'own', "shou

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
#加载名为 'qprop_corpus.txt' 的语料库文件，并将其内容存储在 doc_content_list 列表中。
#dataset = 'qprop_small'
dataset = 'ptc'
doc_content_list = []
f = open('../../data_tgcn/'+dataset+"/"+ dataset + '_corpus.txt', 'rb')#corpus
# f = open('data/wiki_long_abstracts_en_text.txt', 'r')
for line in f.readlines():
    doc_content_list.append(line.strip().decode('latin1'))# 将每一行解码为字符串并存入列表
f.close()


In [3]:
#计算文档集合中每个单词的频率，并筛选掉频率较低的单词，然后将文档内容进行清洗和处理，将每个文档中的单词保存到一个集合中。
word_freq = {}  # to remove rare words

for doc_content in doc_content_list:
    temp = clean_str(doc_content)
    words = temp.split()
    for word in words:
        if word in word_freq:
            word_freq[word] += 1
        else:
            word_freq[word] = 1

clean_docs = []
words_set=set()

In [4]:
for doc_content in doc_content_list:
    words=doc_content.split()
    for word in words:
        words_set.add(word)
print("there are {} words in this dataset.".format(len(words_set)))

for doc_content in doc_content_list:
    temp = clean_str(doc_content)
    words = temp.split()
    doc_words = []
    for word in words:
        if word not in stop_words and word_freq[word] >=5:
            word=poter_stemme.stem(word)# 提取词干
            doc_words.append(word)# 添加到单词列表中

    doc_str = ' '.join(doc_words).strip() # 将单词列表转换为字符串
    clean_docs.append(doc_str)

clean_corpus_str = '\n'.join(clean_docs)

f = open(f'../../data_tgcn/{dataset}/{dataset}.clean.txt', 'w')
#f = open('data/wiki_long_abstracts_en_text.qprop_small_ori.txt', 'w')
f.write(clean_corpus_str)
f.write("\n")
f.close()


there are 37470 words in this dataset.


In [5]:
#这段代码是对已经处理过的文本数据进行统计分析，包括计算文本的最小长度、最大长度和平均长度
#dataset = '20ng'
min_len = 10000
aver_len = 0
max_len = 0 

f = open(f'../../data_tgcn/{dataset}/{dataset}.clean.txt', 'r')
#f = open('data/wiki_long_abstracts_en_text.txt', 'r')
lines = f.readlines()
for line in lines:
    line = line.strip()
    temp = line.split()
    aver_len = aver_len + len(temp)
    if len(temp) < min_len:
        min_len = len(temp)
    if len(temp) > max_len:
        max_len = len(temp)
f.close()
aver_len = 1.0 * aver_len / len(lines)
print('min_len : ' + str(min_len))
print('max_len : ' + str(max_len))
print('average_len : ' + str(aver_len))

min_len : 1
max_len : 65
average_len : 14.39092255607279


In [6]:

f_clean=open("../../data_tgcn/"+dataset+"/"+dataset+".clean.txt","r",encoding="utf-8").readlines()
f_clean_new=open("../../data_tgcn/"+dataset+"/"+dataset+"_new.clean.txt","w",encoding="utf-8")
f_label=open("../../data_tgcn/"+dataset+"/"+dataset+"_labels.txt","r",encoding="utf-8").readlines()
f_label_new=open("../../data_tgcn/"+dataset+"/"+dataset+"_new_labels.txt","w",encoding="utf-8")

In [7]:

ordinary_train_dict={}
ordinary_test_dict={}
i=-1
for line_corpus in f_clean:
    i=i+1
    if len(line_corpus.split(" ")) > 2:# 过滤掉短文本
        mode = f_label[i].split("\t")[1]# 获取训练或测试标签
        label = f_label[i].split("\t")[-1].replace("-1","0") # 获取标签并将“-1”替换为“0”
        if mode=="train":
            ordinary_train_dict[line_corpus]=mode+"\t"+str(label)
        else:
            ordinary_test_dict[line_corpus] = mode + "\t" + str(label)


In [8]:
import random

# 获取训练集字典的所有键（文本内容）
dict_key_ls = list(ordinary_train_dict.keys())
new_train_dict = {}
i = 0

# 按顺序遍历训练集字典的所有键（文本内容），并重新编号
for key in dict_key_ls:
    label = ordinary_train_dict.get(key)
    new_train_dict[key] = str(i) + "\t" + label
    i = i + 1

# 获取测试集字典的所有键（文本内容）
dict_key_ls = list(ordinary_test_dict.keys())
new_test_dict = {}

# 按顺序遍历测试集字典的所有键（文本内容），并继续编号
for key in dict_key_ls:
    label = ordinary_test_dict.get(key)
    new_test_dict[key] = str(i) + "\t" + label
    i = i + 1

# 输出训练集和测试集的总数据量
print(len(new_train_dict) + len(new_test_dict))

# 将重新编号的训练集和测试集数据写入文件
for k, v in new_train_dict.items():
    f_clean_new.write(k)
    f_label_new.write(v)
for k, v in new_test_dict.items():
    f_clean_new.write(k)
    f_label_new.write(v)


17752


In [9]:
# #将训练集和测试集的数据进行随机打乱，并重新编号后保存到新的文件中
# import random
# dict_key_ls=list(ordinary_train_dict.keys())
# #random.shuffle(dict_key_ls)
# new_train_dict={}
# i=0
# for key in dict_key_ls:
#     label=ordinary_train_dict.get((key))
#     new_train_dict[key]=str(i)+"\t"+label
#     i=i+1
    
# dict_key_ls=list(ordinary_test_dict.keys())
# new_test_dict={}
# for key in dict_key_ls:
#     label=ordinary_test_dict.get((key))
#     new_test_dict[key]=str(i)+"\t"+label
#     i=i+1

# print(len(new_train_dict)+len(new_test_dict))
# for k,v in new_train_dict.items():
#     f_clean_new.write(k)
#     f_label_new.write(v)
# for k,v in new_test_dict.items():
#     f_clean_new.write(k)
#     f_label_new.write(v)

# corpus=[]
# labels=[]
# i=-1
# for line_corpus in f_clean:
#     i = i + 1
#     if len(line_corpus.split(" "))>2:
#         if line_corpus not in corpus:
#             corpus.append(line_corpus)
#             # f_clean_new.write(line_corpus)
#             mode=f_label[i].split("\t")[1]
#             label=f_label[i].split("\t")[-1]
#             labels.append(mode+"\t"+label)
#             # f_label_new.write(mode+"\t"+label)
